# Retrieval quality — does memory find genuinely similar past attacks?

Charts the evidence behind three memory-layer decisions, using the **real**
memory module — the same `case_text.render()` and the same pinned embedding
model that Arm B retrieves with. If the template or the model changes,
re-running this notebook shows the effect.

| Figure | Question | Decision it supports |
|---|---|---|
| 1 | Does every case text fit the model's 256-token limit? | BL-013, C3 |
| 2 | How often is the nearest past alert the same attack family? | BL-014, C1 |
| 3 | Can a similarity threshold separate good matches from bad? | C8 (no cut-off) |
| 4 | Which families does retrieval work for? | BL-014 |
| 5 | Are the 2nd and 3rd matches as good as the 1st? | C8 (k = 3) |
| 6 | In the live runs, were the cases the LLM saw right? | BL-118 |
| 7 | Did memory change the verdicts? Three arms, same alerts, same model | BL-118, BL-003 |
| 8 | Does the effect build up as memory fills? | BL-118 |

**Ground truth is read here, after the fact, for evaluation only** — the
legitimate use in `docs/memory-layer/04-read-path.md` (Retrieval quality
audit). It never enters the case text or the store. Nothing in this notebook
writes to a memory store.

**The case template was fixed before this notebook was first run.** These
numbers are reported, not tuned against (CLAUDE.md, Experimental Integrity).

Figures are also saved as PNG to `artifacts/figures/`.

### Kernel: run this on `Python (memsoc-memory)`, not the default `python3`

The embedding model and the store are installed **only** in the memory
virtualenv (`requirements/memory.txt`; it is kept apart from the detector's
environment on purpose, see BL-107). The default `python3` kernel is the system
Python, which has neither, so the first cell would fail. Register the venv as a
kernel once per machine:

```bash
"<HOME>/.venvs/memsoc-memory/Scripts/python" -m ipykernel install --user --name memsoc-memory --display-name "Python (memsoc-memory)"
```

Then pick **Python (memsoc-memory)** as the kernel (VS Code: *Select Kernel*,
top right). To run it headlessly instead:

```bash
jupyter nbconvert --to notebook --execute notebooks/retrieval-quality.ipynb --output-dir artifacts/figures
```

## Results — run of 2026-10-15

**In plain words: memory is only as good as what it stores.**

- **The model's own verdicts:** with these in memory (B-self), the model repeated its own mistake on every alert and grew *more* confident in it.
- **Correct labels:** the same memory, retrieving the same past alerts but holding correct labels (B-oracle), took the model from catching **1%** of real attacks to **88%**.
- **Retrieval:** it finds the right kind of attack about twice as often as chance, but not on the alerts where the detector itself is wrong.

The numbers below were written down from the executed run. They are not regenerated when the notebook is re-run; the cells below are.

### Setup

| | |
|---|---|
| LLM | `gemini-3.5-flash-lite`, temperature 0. **A stand-in:** the decided `gemini-3.8-flash` returned 503 "high demand" all day (BL-117) |
| Stream | stream v2, 120,000 held-out flows, seed 42 → 1,135 flagged alerts, **all real attacks** (the detector raised no false alarms) |
| Memory | template 1.0, MiniLM, top 3, no cut-off, write-all, a fresh store per run |
| Runs | `armA-flashlite-20261015`, `bself-flashlite-20261015`, `boracle-flashlite-20261015`; run side by side, same window |
| Scored | **165 alerts triaged by all three arms**: the first 165 in stream order. The runs stopped at Google's free-tier quota, **500 requests per day per model** |
| Scoring | `score_verdicts.score()`, unchanged; 95% Wilson intervals; exact McNemar paired test |

### Retrieval

| | Overall | Hard cases (detector named the wrong family) |
|---|---|---|
| Offline, all 1,135 attacks: nearest *past* alert same family | **58.7%** | **15.2%** (n=282) |
| Random past attack | 27.5% | 23.3% |
| Live run, top-1 retrieved case same family (n=166) | **56.6%** | **10.3%** (n=29) |
| Random earlier attack in the run | 26.6% | 21.4% |

- **Coverage:** 99.4% of alerts retrieved at least one past case.
- **Cold vs warm:** retrieval improves a little as memory fills, from 53.5% in the first 100 alerts to 61.2% after them.
- **Stored label right:** B-self **0%**, because every verdict it stored was "benign"; B-oracle **56.1%**.

### Verdicts (165 paired alerts)

| Arm | Real attack escalated | Attack family correct | Mean confidence |
|---|---|---|---|
| A (memory off) | 0.6% [0.1–3.4] | 0.6% [0.1–3.4] | 0.88 |
| B-self | 0.0% [0.0–2.3] | 0.0% [0.0–2.3] | 0.94 |
| B-oracle | **88.5%** [82.7–92.5] | **67.3%** [59.8–74.0] | 0.92 |
| *Detector, same alerts* | — | *82.4%* | — |

Paired McNemar against Arm A ("A right, B wrong" vs "A wrong, B right"):
- **B-self:** 1 vs 0, p = 1. No detectable difference.
- **B-oracle, escalation:** 0 vs 145, p = 4.5e-44.
- **B-oracle, family:** 0 vs 110, p = 1.5e-33.

Integrity checks passed:
- All three arms used one model.
- Arm A retrieved nothing and created no store.
- B-self and B-oracle retrieved the identical past alerts for 165 of 165 alerts.

### What this shows

1. **Error propagation is real and measurable** (research question 4).
   - B-self stored 165 wrong "benign" verdicts, so every later alert was shown only wrong precedent.
   - The model's family matched the top case 100% of the time, and it never recovered.
   - Its mean stated confidence rose from 0.88 (Arm A) to 0.94.
   - The one attack Arm A escalated, B-self dismissed.
   - The confidence rise is descriptive; no test for it was planned in advance.
2. **What memory stores matters more than what it retrieves.** Both memory arms retrieved the same cases; only the labels differed. The result was 0% vs 88.5% of attacks escalated.
3. **Even perfect memory did not beat the detector** on naming the family (67.3% vs 82.4%).
4. **Retrieval is weakest where it is needed most.** On hard cases it is worse than random, so memory cannot rescue the detector's own mistakes through similarity.

### Limits — keep these with the numbers

- **Floor effect.** The stand-in model dismisses almost every real attack on its own (Arm A 0.6%), so this run cannot show B-self *improving* on Arm A. Repeat it on `gemini-3.8-flash` before drawing a conclusion about deployable memory.
- **B-oracle is a declared ceiling**, built from true labels; it is not something a SOC could deploy. Its stored cases also look different to the LLM: confidence 1.00 and no reasoning text (BL-113). Part of its effect may come from that difference.
- **Small slice.** 165 of 1,135 alerts (14.5%), all from the start of the stream, so memory never held more than 165 cases.
- **One run per arm**, so there is no run-to-run variance yet; decisions.md B3 asks for three runs.
- **The README's primary metric is not computable on this population.** Escalation precision needs benign alerts (BL-016), and must-escalate recall needs the tier split (BL-105, BL-106).
- Nothing was tuned after seeing these results.

Executed copy with all outputs: `artifacts/figures/retrieval-quality.ipynb`. Figures: `artifacts/figures/01`–`08-*.png`.

In [ ]:
# Setup. The notebook runs from notebooks/, so put the repo's src/ on the path.
import csv
import json
import os
import sys
from collections import Counter
from importlib.util import find_spec
from pathlib import Path

missing = [m for m in ("sentence_transformers", "chromadb", "matplotlib") if find_spec(m) is None]
if missing:
    raise RuntimeError(
        f"This kernel ({sys.executable}) is missing {missing}. Switch the kernel to "
        "'Python (memsoc-memory)' -- see the 'Kernel' note at the top of this notebook."
    )

root = Path.cwd()
while not (root / "src" / "memsoc").is_dir():
    if root.parent == root:
        raise RuntimeError("run this notebook from inside the Mem-SOC repository")
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import matplotlib.pyplot as plt

from memsoc import paths
from memsoc.memory import case_text, embedding

os.makedirs(paths.FIGURES_DIR, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def save(fig, name):
    path = os.path.join(paths.FIGURES_DIR, name)
    fig.savefig(path, bbox_inches="tight")
    print("saved", path)

print("schema_version", case_text.SCHEMA_VERSION, "| model", embedding.MODEL_NAME, "@", embedding.MODEL_REVISION[:10])

## Load the alerts

Every detector-flagged alert in stream order, one per `uid` (the same
de-duplication `triage_runner.load_alerts()` applies), joined to its flow
features, then to its true label.

In [ ]:
alerts = {}
with open(paths.LIVE_RESULTS, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            if record.get("alert") and record["uid"] not in alerts:
                alerts[record["uid"]] = record

flows = {}
with open(paths.GENERATED_LOGS, encoding="utf-8") as f:
    for line in f:
        flow = json.loads(line)
        if flow["uid"] in alerts:
            flows[flow["uid"]] = flow

with open(paths.GROUND_TRUTH, encoding="utf-8", newline="") as f:
    truth = {row["uid"]: (int(row["label"]), row["attack"]) for row in csv.DictReader(f)}

uids = list(alerts)                       # stream order
missing = [u for u in uids if u not in flows or u not in truth]
assert not missing, f"{len(missing)} alerts could not be joined"

is_attack = np.array([truth[u][0] == 1 for u in uids])
family = np.array([truth[u][1] for u in uids])
detector = np.array([alerts[u]["multiclass"]["prediction"] for u in uids])

print(f"{len(uids)} detector-flagged alerts: {is_attack.sum()} real attacks, {(~is_attack).sum()} false positives")
print("families:", {str(k): v for k, v in Counter(family[is_attack]).most_common()})

## Figure 1 — Do the case texts fit the model?

MiniLM truncates at 256 tokens **without raising**, so an over-long case would
embed as if its tail did not exist (BL-013). The memory module now raises
instead; this shows how much headroom template 1.0 has.

In [ ]:
texts = [case_text.render(alerts[u], flows[u]) for u in uids]
tokens = np.array([embedding.token_count(t) for t in texts])
limit = embedding.max_tokens()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(tokens, bins=20, color="#4C72B0")
ax.axvline(limit, color="#C44E52", linestyle="--", label=f"model limit ({limit})")
ax.set_xlim(0, limit + 20)
ax.set_xlabel("tokens per case text (model's own tokenizer)")
ax.set_ylabel("alerts")
ax.set_title(f"Case text length: p50 {np.median(tokens):.0f}, p99 {np.percentile(tokens, 99):.0f}, max {tokens.max()}")
ax.legend()
save(fig, "01-token-lengths.png")
plt.show()

print("example case text:\n")
print(texts[0])

## Embed every case, and find each attack's nearest neighbours

Only real attacks are compared — a false positive has no attack family to
match. **Hard** cases are those where the detector named the wrong family:
the population memory exists to help with.

Two views of "nearest":

- **any other** — compared against every other attack, as the investigation did.
- **past only** — compared only against attacks *earlier in the stream*. This
  is what memory can actually see during a run. The first attack has no past
  and is left out of this view.

In [ ]:
idx = np.flatnonzero(is_attack)
E = np.array([embedding.embed(texts[i]) for i in idx])
S = E @ E.T                                  # cosine similarity: vectors are unit length
fam = family[idx]
hard = detector[idx] != fam
n = len(idx)

def neighbours(past_only, k):
    """Top-k neighbour positions for each attack, most similar first; None if no candidates."""
    out = []
    for i in range(n):
        allowed = np.arange(i) if past_only else np.delete(np.arange(n), i)
        if len(allowed) == 0:
            out.append(None)
            continue
        order = allowed[np.argsort(-S[i, allowed], kind="stable")]
        out.append(order[:k])
    return out

nn_any = neighbours(False, 3)
nn_past = neighbours(True, 3)

def match_rate(nbrs, mask, k=1):
    """Mean share of the top-k neighbours with the same family, over `mask`."""
    hits, total = 0.0, 0
    for i in np.flatnonzero(mask):
        if nbrs[i] is None:
            continue
        top = nbrs[i][:k]
        hits += np.mean(fam[top] == fam[i])
        total += 1
    return hits / total, total

def random_rate(mask):
    """Chance that a uniformly random OTHER attack shares the family."""
    counts = Counter(fam)
    rates = [(counts[fam[i]] - 1) / (n - 1) for i in np.flatnonzero(mask)]
    return float(np.mean(rates))

everyone = np.ones(n, dtype=bool)
print(f"{n} attacks, {hard.sum()} hard (detector named the wrong family)")

## Figure 2 — How often is the nearest past alert the same attack family?

The headline retrieval number. The **random** bar is the chance a randomly
chosen other attack shares the family; the **detector** bar is how often the
frozen detector itself names the family correctly. For memory to help on hard
cases, retrieval has to beat random there.

In [ ]:
groups = [("all attacks", everyone), ("hard cases", hard)]
series = {
    "nearest (any other)": [match_rate(nn_any, m) for _, m in groups],
    "nearest (past only)": [match_rate(nn_past, m) for _, m in groups],
    "random other attack": [(random_rate(m), int(m.sum())) for _, m in groups],
    "detector accuracy":   [(float(np.mean(detector[idx][m] == fam[m])), int(m.sum())) for _, m in groups],
}
colours = ["#4C72B0", "#55A868", "#8C8C8C", "#C44E52"]

fig, ax = plt.subplots(figsize=(8, 3.8))
width = 0.2
x = np.arange(len(groups))
for j, ((name, values), colour) in enumerate(zip(series.items(), colours)):
    rates = [v[0] for v in values]
    bars = ax.bar(x + (j - 1.5) * width, rates, width, label=name, color=colour)
    for bar, (rate, total) in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.01, f"{rate:.0%}\nn={total}",
                ha="center", va="bottom", fontsize=7)
ax.set_xticks(x, [g for g, _ in groups])
ax.set_ylim(0, 1.25)
ax.set_ylabel("same attack family")
ax.set_title("Nearest-neighbour family match (template 1.0)")
ax.legend(fontsize=8, ncol=2, loc="upper right")
save(fig, "02-family-match.png")
plt.show()

for name, values in series.items():
    print(f"{name:<22}", "  ".join(f"{g}: {r:.1%} (n={t})" for (g, _), (r, t) in zip(groups, values)))

## Figure 3 — Can a similarity threshold separate good matches from bad?

If correct and wrong nearest neighbours had clearly different similarity
scores, a cut-off could drop the bad ones. Heavily overlapping distributions
mean no threshold works — the evidence for C8's "no cut-off".

In [ ]:
best = np.array([S[i, nn_any[i][0]] for i in range(n)])
correct = np.array([fam[nn_any[i][0]] == fam[i] for i in range(n)])

fig, ax = plt.subplots(figsize=(7, 3.2))
bins = np.linspace(best.min(), 1.0, 30)
ax.hist(best[correct], bins=bins, alpha=0.65, label=f"same family (n={correct.sum()})", color="#55A868")
ax.hist(best[~correct], bins=bins, alpha=0.65, label=f"different family (n={(~correct).sum()})", color="#C44E52")
ax.set_xlabel("cosine similarity to nearest other attack")
ax.set_ylabel("attacks")
ax.set_title(f"Median similarity: correct {np.median(best[correct]):.3f} vs wrong {np.median(best[~correct]):.3f}")
ax.legend()
save(fig, "03-similarity-separation.png")
plt.show()

## Figure 4 — Which families does retrieval work for?

Match rate of the nearest other attack, per true family. Small families have
few possible matches, so read the `n` before the bar.

In [ ]:
families = [f for f, _ in Counter(fam).most_common()]
rates = [match_rate(nn_any, fam == f)[0] for f in families]
counts = [int((fam == f).sum()) for f in families]
hard_counts = [int((hard & (fam == f)).sum()) for f in families]

fig, ax = plt.subplots(figsize=(10, 3.8))
bars = ax.bar(families, rates, color="#4C72B0")
for bar, rate, total, h in zip(bars, rates, counts, hard_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.02, f"{rate:.0%}\nn={total}, hard={h}",
            ha="center", va="bottom", fontsize=7)
ax.set_ylim(0, 1.25)
ax.set_ylabel("nearest neighbour same family")
ax.set_title("Family match by true attack family")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
save(fig, "04-match-by-family.png")
plt.show()

## Figure 5 — Are the 2nd and 3rd matches as good as the 1st?

Arm B retrieves 3 cases (C8). This shows the share of the top *k* that share
the alert's family, for k = 1, 2, 3.

In [ ]:
ks = [1, 2, 3]
fig, ax = plt.subplots(figsize=(6, 3.4))
for (label, mask), colour in zip(groups, ["#4C72B0", "#C44E52"]):
    for nbrs, style, view in [(nn_any, "-", "any other"), (nn_past, "--", "past only")]:
        values = [match_rate(nbrs, mask, k)[0] for k in ks]
        ax.plot(ks, values, style, marker="o", color=colour, label=f"{label}, {view}")
        print(f"{label:<12} {view:<10}", "  ".join(f"k={k}: {v:.1%}" for k, v in zip(ks, values)))
ax.set_xticks(ks)
ax.set_xlabel("k (cases retrieved)")
ax.set_ylabel("share of top-k with same family")
ax.set_ylim(0, 1)
ax.set_title("Top-k family precision")
ax.legend(fontsize=8)
save(fig, "05-top-k-precision.png")
plt.show()

## Live runs: what memory actually handed the LLM

Figures 1–5 are offline: they ask what retrieval *would* return. From here on
the notebook reads real triage runs (`triage_runner --memory <condition>
--run-id <id>`) and asks what retrieval *did* return, and what the LLM did
with it. Each run's verdict log is `artifacts/runs/<id>/triage_verdicts.jsonl`;
a memory run's stored cases are in `artifacts/memory/<id>/cases.jsonl`.

The three arms (decisions.md A3) replay the **same alerts, in the same order,
on the same model, in the same time window**:

| Arm | Memory | What the past cases tell the LLM |
|---|---|---|
| A | off | nothing: no past cases are shown |
| B-self | on | the LLM's own earlier verdicts, right or wrong |
| B-oracle | on | the true labels; a declared ceiling, never the headline |

B-self and B-oracle retrieve the **same** past alerts (the embedded text is
identical); only the stored label differs. Comparing them separates the
question "does retrieval find useful cases?" from "is what memory stores
correct?".

For every retrieved case, two different "right" answers are measured:

- **same true family**: the past alert really was the same kind of attack.
  This depends only on retrieval, so it is the same in both memory arms.
- **stored label right**: the label memory *showed* the LLM for that case
  names the current alert's true family. This is what the LLM was actually
  told. In B-self, a wrong stored label is where error propagation starts.

**Cold** is the first 100 alerts of a run, when memory is nearly empty;
**warm** is the rest. Ground truth is read here after the runs, for scoring
only.

In [ ]:
# Which three-arm run to read. Each tag is one model; never mix tags.
#   "flashlite-20261015" -- gemini-3.5-flash-lite stand-in (Results, 2026-10-15)
#   protocol v3 (docs/status/testing-protocol-v3.md): "gptoss20b-p3", "qwen38-27b-p3", "gemma4-26b-p3"
#   the p1 and p2 run ids exist only as superseded partial runs (BL-125, BL-126); never pool them with p3
RUN_TAG = "gemma4-26b-p3"

RUNS = {
    "A (memory off)": f"armA-{RUN_TAG}",
    "B-self": f"bself-{RUN_TAG}",
    "B-oracle": f"boracle-{RUN_TAG}",
}
COLD = 100


def read_run(run_id):
    """A run's verdict rows in stream order, one per uid; [] if the run does not exist."""
    path = Path(paths.run_dir(run_id)) / "triage_verdicts.jsonl"
    if not path.exists():
        return []
    rows = {}
    for line in open(path, encoding="utf-8"):
        if line.strip():
            row = json.loads(line)
            rows[row["uid"]] = row
    return sorted(rows.values(), key=lambda r: r["seq"])


def read_cases(run_id):
    """case_id -> (alert uid, stored attack_category), from a memory run's sidecar."""
    path = Path(paths.memory_store(run_id)) / "cases.jsonl"
    cases = {}
    if path.exists():
        for line in open(path, encoding="utf-8"):
            if line.strip():
                c = json.loads(line)
                cases[c["case_id"]] = (c["metadata"]["alert_id"], c["metadata"]["attack_category"])
    return cases


def live_items(arm):
    """One entry per real attack the run triaged: what was retrieved, and whether it was right."""
    cases = read_cases(RUNS[arm])
    items = []
    for rank, row in enumerate(runs[arm]):
        label, true_family = truth[row["uid"]]
        if label != 1:
            continue
        used = [c for c in row["precedent_used"].split(",") if c]
        items.append({
            "rank": rank,
            "uid": row["uid"],
            "family": true_family,
            "hard": row["detector_label"] != true_family,
            "used": used,
            "same_family": [truth[cases[c][0]][1] == true_family for c in used],
            "stored_ok": [cases[c][1] == true_family for c in used],
            "top_label": cases[used[0]][1] if used else None,
        })
    return items


runs = {arm: read_run(run_id) for arm, run_id in RUNS.items()}
for arm, rows in runs.items():
    model = rows[0]["model_id"] if rows else "-"
    print(f"{arm:<16} {RUNS[arm]:<28} {len(rows):>5} alerts triaged   model {model}")

live = {arm: live_items(arm) for arm in ("B-self", "B-oracle") if runs[arm]}

In [ ]:
def live_rate(items, key, k):
    """Mean share of the top-k retrieved cases that are right, over alerts that retrieved any."""
    scored = [np.mean(a[key][:k]) for a in items if a["used"]]
    return (float(np.mean(scored)) if scored else float("nan")), len(scored)


def live_random(items):
    """Chance that a random EARLIER attack in the same run shares the alert's family."""
    rates = []
    for a in items:
        if a["used"]:
            past = [b["family"] for b in items if b["rank"] < a["rank"]]
            if past:
                rates.append(np.mean([f == a["family"] for f in past]))
    return (float(np.mean(rates)) if rates else float("nan")), len(rates)


def live_groups(items):
    return [
        ("all attacks", items),
        ("hard cases", [a for a in items if a["hard"]]),
        (f"cold (first {COLD})", [a for a in items if a["rank"] < COLD]),
        ("warm", [a for a in items if a["rank"] >= COLD]),
    ]


for arm, items in live.items():
    retrieved_any = np.mean([bool(a["used"]) for a in items])
    print(f"\n{arm}: alerts that retrieved at least one past case: {retrieved_any:.1%}")
    print(f"{'':<18}{'same family top-1':>18}{'top-3':>8}{'stored label top-1':>20}{'top-3':>8}{'random':>8}")
    for name, group in live_groups(items):
        f1, n1 = live_rate(group, "same_family", 1)
        f3, _ = live_rate(group, "same_family", 3)
        s1, _ = live_rate(group, "stored_ok", 1)
        s3, _ = live_rate(group, "stored_ok", 3)
        r, _ = live_random(group)
        print(f"{name:<18}{f1:>18.1%}{f3:>8.1%}{s1:>20.1%}{s3:>8.1%}{r:>8.1%}   n={n1}")

## Figure 6 — Live runs: were the retrieved cases right?

Top-1 case per alert. Blue is retrieval alone (the same in both memory arms);
red and green are whether the label memory showed was right; grey is the
chance a random earlier attack shares the family — the bar retrieval has to
beat.

In [ ]:
if live:
    first = next(iter(live))
    names = [name for name, _ in live_groups(live[first])]
    series6 = {"same true family (retrieval)": [live_rate(g, "same_family", 1) for _, g in live_groups(live[first])]}
    for arm in live:
        series6[f"stored label right ({arm})"] = [live_rate(g, "stored_ok", 1) for _, g in live_groups(live[arm])]
    series6["random earlier attack"] = [live_random(g) for _, g in live_groups(live[first])]
    colours = {"same true family (retrieval)": "#4C72B0", "stored label right (B-self)": "#C44E52",
               "stored label right (B-oracle)": "#55A868", "random earlier attack": "#8C8C8C"}

    fig, ax = plt.subplots(figsize=(10, 4))
    width = 0.8 / len(series6)
    x = np.arange(len(names))
    for j, (label, values) in enumerate(series6.items()):
        bars = ax.bar(x + (j - (len(series6) - 1) / 2) * width, [v[0] for v in values], width,
                      label=label, color=colours[label])
        for bar, (rate, total) in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.01, f"{rate:.0%}",
                    ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x, names)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("top-1 retrieved case")
    ax.set_title("Live runs: retrieval as the LLM saw it")
    ax.legend(fontsize=8, ncol=2, loc="upper right")
    save(fig, "06-live-run-retrieval.png")
    plt.show()

## Three-arm comparison: did memory change the verdicts?

Only alerts triaged by **all three** arms are compared, so every difference is
paired: the same alert, the same model, memory being the only thing that
changed.

Scoring uses the project's own harness, `memsoc.triage.score_verdicts.score()`,
imported unchanged; nothing is redefined here. Two outcomes per alert:

- **real attack escalated**: the verdict was `true_positive`. Every alert in
  this stream is a real attack (the detector raised no false alarms on it), so
  `false_positive` means a real attack was dismissed, and `inconclusive` means
  no call was made.
- **attack family correct**: the LLM named the true family.

The README's primary metric, escalation precision with must-escalate recall,
cannot be computed yet: precision needs benign alerts in the population
(BL-016) and must-escalate recall needs the tier split (BL-105, BL-106).
These two outcomes are what this population can honestly measure.

Each rate carries a 95% Wilson interval. Arms are compared with an exact
McNemar test on the paired right/wrong outcomes: *b* counts alerts Arm A got
right and the memory arm got wrong, *c* the reverse. Only *b* and *c* carry
information; a small p says memory changed outcomes more in one direction
than chance would.

In [ ]:
import math

from memsoc.triage import score_verdicts

gt_rows = score_verdicts.read_ground_truth(paths.GROUND_TRUTH)

completed = [{r["uid"] for r in rows} for rows in runs.values()]
paired = set.intersection(*completed) if all(completed) else set()
by_arm = {arm: {r["uid"]: r for r in rows if r["uid"] in paired} for arm, rows in runs.items()}
order = sorted(paired, key=lambda u: by_arm["A (memory off)"][u]["seq"])


def family_ok(row):
    return row["attack_category"] == score_verdicts.expected_category(gt_rows[row["uid"]])


def wilson(k, n, z=1.96):
    """Rate with its 95% Wilson score interval."""
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return p, centre - half, centre + half


def mcnemar(a_ok, b_ok):
    """Exact two-sided McNemar test. b = A right & B wrong, c = A wrong & B right."""
    b = sum(x and not y for x, y in zip(a_ok, b_ok))
    c = sum(y and not x for x, y in zip(a_ok, b_ok))
    m = b + c
    if m == 0:
        return b, c, 1.0
    tail = sum(math.comb(m, k) for k in range(min(b, c) + 1)) / 2 ** m
    return b, c, min(1.0, 2 * tail)


outcome = {}
if order:
    print(f"{len(order)} alerts triaged by all three arms (paired)\n")
    print(f"{'':<16}{'real attack escalated':>28}{'attack family correct':>28}"
          f"{'dismissed':>11}{'no call':>9}{'= detector':>12}{'confidence':>12}")
    for arm in RUNS:
        rows = [by_arm[arm][u] for u in order]
        s = score_verdicts.score(rows, gt_rows)
        outcome[arm] = {
            "score": s,
            "escalated": [r["verdict"] == "true_positive" for r in rows],
            "family": [family_ok(r) for r in rows],
            "confidence": [float(r["confidence"]) for r in rows],
        }
        e = wilson(sum(outcome[arm]["escalated"]), len(rows))
        f = wilson(sum(outcome[arm]["family"]), len(rows))
        print(f"{arm:<16}{e[0]:>12.1%} [{e[1]:5.1%}, {e[2]:5.1%}]{f[0]:>12.1%} [{f[1]:5.1%}, {f[2]:5.1%}]"
              f"{s['fn']:>11}{s['inconclusive']:>9}{s['agreement']:>12.1%}{np.mean(outcome[arm]['confidence']):>12.2f}")

    detector_accuracy = outcome["A (memory off)"]["score"]["det_accuracy"]
    print(f"\ndetector's own family accuracy on these alerts: {detector_accuracy:.1%}")

    print("\nPaired exact McNemar test against Arm A:")
    for arm in ("B-self", "B-oracle"):
        for metric, label in (("escalated", "escalated"), ("family", "family correct")):
            b, c, p = mcnemar(outcome["A (memory off)"][metric], outcome[arm][metric])
            print(f"  {arm:<9} {label:<15} A right, {arm} wrong: {b:<5} A wrong, {arm} right: {c:<5} p = {p:.2g}")
else:
    print("no alert has been triaged by all three arms yet")

### Integrity checks

These must hold for the comparison to mean anything: one model in every arm,
Arm A never touching memory, and the two memory arms retrieving the same
past alerts so that only the stored labels differ.

In [ ]:
if order:
    models = {r["model_id"] for rows in runs.values() for r in rows}
    print("model used by every arm:", models)
    assert len(models) == 1, "arms ran on different models -- not comparable"

    arm_a = runs["A (memory off)"]
    assert all(r["precedent_count"] == 0 and r["precedent_used"] == "" for r in arm_a)
    assert not os.path.exists(paths.memory_store(RUNS["A (memory off)"])), "Arm A created a memory store"
    print("Arm A retrieved nothing and created no memory store")

    def positions(rows):
        return {r["uid"]: [c.split(":")[-1] for c in r["precedent_used"].split(",") if c] for r in rows}

    self_pos, oracle_pos = positions(runs["B-self"]), positions(runs["B-oracle"])
    common = self_pos.keys() & oracle_pos.keys()
    same = sum(self_pos[u] == oracle_pos[u] for u in common)
    print(f"B-self and B-oracle retrieved the same past alerts, in the same order, for {same} of {len(common)} alerts")

## Figure 7 — Three arms, same alerts, same model

Error bars are 95% Wilson intervals. The dashed line is the frozen detector's
own family accuracy on the same alerts.

In [ ]:
if outcome:
    arms = list(outcome)
    fig, ax = plt.subplots(figsize=(8, 3.8))
    width = 0.35
    x = np.arange(len(arms))
    for j, (metric, label, colour) in enumerate([
        ("escalated", "real attack escalated", "#4C72B0"),
        ("family", "attack family correct", "#55A868"),
    ]):
        stats = [wilson(sum(outcome[a][metric]), len(outcome[a][metric])) for a in arms]
        rates = [s[0] for s in stats]
        err = [[s[0] - s[1] for s in stats], [s[2] - s[0] for s in stats]]
        bars = ax.bar(x + (j - 0.5) * width, rates, width, yerr=err, capsize=3, label=label, color=colour)
        for bar, rate in zip(bars, rates):
            ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.05, f"{rate:.0%}", ha="center", fontsize=8)
    ax.axhline(detector_accuracy, color="#C44E52", linestyle="--", linewidth=1,
               label=f"detector family accuracy ({detector_accuracy:.0%})")
    ax.set_xticks(x, arms)
    ax.set_ylim(0, 1.2)
    ax.set_ylabel(f"share of {len(order)} paired alerts")
    ax.set_title("Did memory change the verdicts? (95% Wilson CI)")
    ax.legend(fontsize=8, loc="upper left")
    save(fig, "07-arm-comparison.png")
    plt.show()

## Figure 8 — Does the effect build up as memory fills?

Rolling mean over 50 consecutive alerts, in stream order. Memory starts empty,
so any effect should be absent at the left and grow; Arm A has no memory and
should stay flat apart from the stream's own drift. The dotted line marks the
end of the cold period.

In [ ]:
if outcome:
    W = min(50, len(order))   # shrinks only while a run is still short

    def rolling(values):
        return np.convolve(np.asarray(values, dtype=float), np.ones(W) / W, mode="valid")

    colours = {"A (memory off)": "#8C8C8C", "B-self": "#C44E52", "B-oracle": "#55A868"}
    panels = [("escalated", "real attack escalated"), ("family", "attack family correct"),
              ("confidence", "LLM's stated confidence")]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
    for ax, (metric, label) in zip(axes, panels):
        for arm, o in outcome.items():
            xs = np.arange(W - 1, len(o[metric]))
            ax.plot(xs, rolling(o[metric]), color=colours[arm], label=arm, linewidth=1.4)
        ax.axvline(COLD, color="k", linewidth=0.6, linestyle=":")
        ax.set_title(f"{label} (rolling {W})", fontsize=10)
        ax.set_xlabel("alert position in the stream")
        ax.set_ylim(-0.02, 1.02)
    axes[0].legend(fontsize=8)
    save(fig, "08-over-time.png")
    plt.show()

## How memory changed individual answers

For each memory arm, against Arm A on the same alert, split by whether the
**top past case's stored label** was right for the current alert:

- **fixed**: Arm A got it wrong, the memory arm got it right.
- **broke**: Arm A got it right, the memory arm got it wrong.

Memory that helps should fix more than it breaks when it shows a right label,
and an LLM that is merely copying memory will break answers whenever the label
shown is wrong: that is error propagation, measured directly.

**Follows memory** is how often the LLM's family equals the top case's stored
label. The same figure for Arm A, on the same alerts, is what agreement would
be anyway without seeing memory; the gap is memory's influence.

In [ ]:
if outcome:
    a = by_arm["A (memory off)"]
    checks = {
        "escalated": lambda row: row["verdict"] == "true_positive",
        "family correct": family_ok,
    }
    for arm in ("B-self", "B-oracle"):
        if arm not in live:
            continue
        items = {i["uid"]: i for i in live[arm]}
        b_rows = by_arm[arm]
        with_top = [u for u in order if u in items and items[u]["used"]]

        follows_b = np.mean([b_rows[u]["attack_category"] == items[u]["top_label"] for u in with_top])
        follows_a = np.mean([a[u]["attack_category"] == items[u]["top_label"] for u in with_top])
        print(f"\n{arm}  (n={len(with_top)} alerts with at least one past case)")
        print(f"  follows memory: {follows_b:.1%} with memory vs {follows_a:.1%} for Arm A on the same alerts")

        for shown_right in (True, False):
            group = [u for u in with_top if items[u]["stored_ok"][0] == shown_right]
            parts = []
            for name, ok in checks.items():
                fixed = sum(not ok(a[u]) and ok(b_rows[u]) for u in group)
                broke = sum(ok(a[u]) and not ok(b_rows[u]) for u in group)
                parts.append(f"{name}: fixed {fixed}, broke {broke}")
            print(f"  top case's stored label {'RIGHT' if shown_right else 'WRONG'} (n={len(group)}):  " + "   ".join(parts))

## Figure 9 — Results at a glance

One frame for the question the experiment asks. Every alert in this stream is
a real attack, so on the left the green share is the only right answer: red is
a real attack waved through as a false alarm, grey is no call made at all.

The middle and right panels count the answers memory **changed**, alert by
alert, against Arm A on the same alert. Green is an answer the memory arm got
right and Arm A got wrong; red is the reverse. Memory that helps fixes more
than it breaks; memory that propagates the model's own errors breaks more than
it fixes.

Nothing new is computed here — every number is from the table above and from
Figures 7 and 8. This cell only puts the answer in one picture, with the plain
wording printed underneath so the colours cannot be misread.


In [ ]:
if outcome:
    PLAIN = {
        "A (memory off)": "no memory",
        "B-self": "memory, own verdicts",
        "B-oracle": "memory, correct labels",
    }
    TICK = {
        "A (memory off)": "A\nno memory",
        "B-self": "B-self\nown verdicts",
        "B-oracle": "B-oracle\ncorrect labels",
    }
    arms = list(outcome)
    model = runs["A (memory off)"][0]["model_id"]

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), gridspec_kw={"width_ratios": [1.6, 1, 1]})

    # Left: what each arm answered. Every alert here is a real attack, so the
    # green share is the only right answer and the other two are both misses.
    verdicts = [
        ("called it a real attack", "#55A868", lambda r: r["verdict"] == "true_positive"),
        ("dismissed it as a false alarm", "#C44E52", lambda r: r["verdict"] == "false_positive"),
        ("made no call", "#8C8C8C", lambda r: r["verdict"] not in ("true_positive", "false_positive")),
    ]
    ax = axes[0]
    y = np.arange(len(arms))[::-1]
    left = np.zeros(len(arms))
    for name, colour, is_kind in verdicts:
        share = np.array([np.mean([is_kind(by_arm[a][u]) for u in order]) for a in arms])
        ax.barh(y, share, 0.5, left=left, color=colour, label=name)
        for yi, (s, l) in zip(y, zip(share, left)):
            if s >= 0.08:
                ax.text(l + s / 2, yi, f"{s:.0%}", ha="center", va="center", color="white", fontsize=9)
        left = left + share
    ax.set_yticks(y, [TICK[a] for a in arms], fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_xticks(np.linspace(0, 1, 6), [f"{t:.0%}" for t in np.linspace(0, 1, 6)], fontsize=8)
    ax.set_xlabel(f"share of the {len(order)} alerts, all of them real attacks")
    ax.set_title("What the LLM answered", fontsize=10)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.40), ncol=3, frameon=False)

    # Middle and right: answers memory changed, alert by alert, against Arm A.
    a_rows = by_arm["A (memory off)"]
    mem_arms = [a for a in ("B-self", "B-oracle") if a in outcome]
    changes = [("called it a real attack", lambda r: r["verdict"] == "true_positive"),
               ("named the right attack family", family_ok)]
    for ax, (title, is_right) in zip(axes[1:], changes):
        x = np.arange(len(mem_arms))
        fixed = [sum(not is_right(a_rows[u]) and is_right(by_arm[arm][u]) for u in order) for arm in mem_arms]
        broke = [-sum(is_right(a_rows[u]) and not is_right(by_arm[arm][u]) for u in order) for arm in mem_arms]
        for counts, name, colour in ((fixed, "memory fixed it", "#55A868"),
                                     (broke, "memory broke it", "#C44E52")):
            ax.bar(x, counts, 0.45, color=colour, label=name)
            for xi, c in zip(x, counts):
                if c:
                    ax.text(xi, c, f" {abs(c)} ", ha="center",
                            va="bottom" if c > 0 else "top", fontsize=9)
        ax.axhline(0, color="k", linewidth=0.8)
        ax.set_xticks(x, [TICK[a] for a in mem_arms], fontsize=9)
        ax.set_xlim(-0.6, len(mem_arms) - 0.4)
        ax.margins(y=0.22)
        ax.set_title(title, fontsize=10)
    axes[1].set_ylabel("alerts changed vs no memory")
    axes[1].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.40), ncol=2, frameon=False)

    fig.suptitle(f"Three arms on the same {len(order)} alerts  |  {model}", fontsize=11)
    save(fig, "09-results-at-a-glance.png")
    plt.show()

    # The same thing in words, so the figure cannot be misread.
    print(f"Of the {len(order)} alerts, every one a real attack:")
    for arm in arms:
        print(f"  {PLAIN[arm]:<24} called {np.mean(outcome[arm]['escalated']):>6.1%} of them real attacks, "
              f"and named the right family for {np.mean(outcome[arm]['family']):>6.1%}")
    for arm in mem_arms:
        f_ = sum(not family_ok(a_rows[u]) and family_ok(by_arm[arm][u]) for u in order)
        b_ = sum(family_ok(a_rows[u]) and not family_ok(by_arm[arm][u]) for u in order)
        print(f"  {PLAIN[arm]:<24} families: fixed {f_}, broke {b_} (net {f_ - b_:+d}) against no memory")

## Reading these results

- Compare **hard cases** against **random** in Figure 2. Retrieval that does
  not beat random on hard cases cannot help the LLM where the detector fails.
- **Past only** is the realistic view: early alerts have little or nothing to
  retrieve, which is the cold start the brief requires reporting separately.
- **B-self vs Arm A** is the headline: memory as it would actually be
  deployed. **B-oracle vs B-self** isolates what the *quality* of stored
  labels is worth, since both retrieve the same cases.
- A null or negative result is a valid outcome (CLAUDE.md, Experimental
  Integrity). Nothing in this notebook was tuned after seeing its numbers.
- "Same family" is a proxy for relevance, not relevance itself. The manual
  audit in `docs/memory-layer/04-read-path.md` answers the stronger question.